# Task B Run 32 — full-data predictions from the Run 30 prompt ensemble

This is the full-data counterpart of Run 30. It trains four Gemma-4-12B QLoRA classifiers:

| prompt | seeds |
|---|---|
| standard label-name prompt | 42, 43 |
| organisers' definitions prompt | 42, 43 |

The recipe matches Run 30: 3 epochs, balanced class weights, rank-16 QLoRA over attention and MLP projections, learning rate 1e-4, and a 320-token cap. Each model trains on all 3,532 deduplicated labelled rows in data/derived/task_b_combined_holdout/all.csv. The four test probability matrices are averaged and converted into one submission.

This notebook produces:

- RECOMMENDED_b32_gemma_prompt_ensemble.zip, containing only predictions.csv
- the averaged and per-prompt test probability matrices
- result.json with model settings and predicted class shares

There is no local F1 score because the released test labels are not available. Use the ZIP for the final Task B submission.

Kaggle settings: Internet on, GPU T4 x2, and a Kaggle secret named HF_TOKEN if Hugging Face asks for authentication. Save Version → Save & Run All. Expected runtime is roughly 4–6 hours.


In [ ]:
import json, os, pathlib, re, shutil, subprocess, sys, time

T_START = time.time()
HARD_STOP = T_START + 11 * 3600
REPO = 'https://github.com/robinpnalex/Hastika-ICON2026.git'
BRANCH = 'task-b'
WORK = '/tmp/hastika'

def git(*args):
    return subprocess.run(['git', '-C', WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

try:
    remote = subprocess.run(['git', 'ls-remote', '--exit-code', REPO, f'refs/heads/{BRANCH}'],
                            check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit('Cannot reach GitHub. Turn on Settings -> Internet, then restart.') from e

if os.path.isdir(WORK + '/.git'):
    git('fetch', '-q', '--depth', '1', 'origin', BRANCH)
    git('reset', '-q', '--hard', 'FETCH_HEAD')
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, '--depth', '1', REPO, WORK], check=True)
head = git('rev-parse', 'HEAD')
assert head == remote, f'clone is at {head[:8]} but {BRANCH} is at {remote[:8]}'
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass
print('HF_TOKEN available:', bool(os.environ.get('HF_TOKEN')))

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'select GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('repo:', git('log', '-1', '--oneline'))
print('gpus:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path('/kaggle/working/b32_prompt_ensemble_full_outputs')
RUNS, LOGS, SUB = OUT / 'runs', OUT / 'logs', OUT / 'submissions'
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = '/tmp/hf'
os.environ['HF_HUB_CACHE'] = HF_CACHE

from hastika.common.gpu_queue import run_queue

MODEL = 'google/gemma-4-12B'
SPLIT = pathlib.Path(WORK) / 'data/derived/task_b_combined_holdout'
TRAIN_CSV = SPLIT / 'all.csv'
LABELS = ['Gender', 'Geo-political', 'Others', 'Political', 'Religion', 'Violence']
LABEL_COL = 'Hate Category'
TEST_CSV = pathlib.Path('/tmp/task_b_test_inputs.csv')
raw_test = pd.read_csv(pathlib.Path(WORK) / 'data/raw/hastika_multiclass_test.csv')
raw_test[['id', 'Comment']].to_csv(TEST_CSV, index=False)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
assert len(train) == 3532, len(train)
assert len(test) == 396, len(test)
assert train['id'].is_unique and test['id'].is_unique
print(f'training on all {len(train)} labelled rows; predicting {len(test)} test rows')


In [ ]:
# Gemma requires a newer transformers/peft stack than the system environment.
LLM_PY = '/tmp/llmenv/bin/python'
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip',
                    '/tmp/llmenv'], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, '-c',
                "import transformers, peft, bitsandbytes; print('LLM env:', transformers.__version__, peft.__version__)"],
               check=True)

def download_model():
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get('HF_TOKEN') or None,
                      allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt',
                                      'tokenizer*', '*.jinja'])

def llm_job(prompt, seed):
    name = f'{prompt}_full_s{seed}'
    cmd = (f'{LLM_PY} -u -m hastika.task_b.llm_classifier --task b --model {MODEL} '
           f'--prompt {prompt} --class-weight balanced --train {TRAIN_CSV} '
           f'--predict {TEST_CSV} --out {RUNS / name} --seed {seed} '
           f'--epochs 3 --lr 1e-4 --r 16 --bs 8 --grad-accum 2 '
           f'--eval-bs 16 --max-len 320 --cache {HF_CACHE}')
    return {'name': name, 'cmd': cmd, 'log': str(LOGS / f'{name}.log')}

def run_jobs(jobs):
    registry = {j['name']: j for j in jobs}
    retried = set()

    def on_done(name, code):
        job = registry[name]
        if not code and name not in retried:
            return []
        log_path = pathlib.Path(job['log'])
        log = log_path.read_text(errors='replace') if log_path.exists() else ''
        extra = ''
        if code == 4 and name not in retried:
            m = re.search(r'--seed (\d+)', job['cmd'])
            if m:
                extra = f' --seed {int(m.group(1)) + 1000}'
                print(f'{name}: collapse detected; retrying with alternate seed')
        elif 'OutOfMemoryError' in log and name not in retried:
            extra = ' --bs 4 --grad-accum 4 --eval-bs 8'
            print(f'{name}: OOM detected; retrying with smaller micro-batches')
        if extra:
            retried.add(name)
            retry = dict(job)
            retry['cmd'] = job['cmd'] + extra
            registry[name] = retry
            return [retry]
        return []
    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=on_done, stop_at=HARD_STOP)

download_model()
jobs = [llm_job(prompt, seed) for prompt in ('short', 'defs') for seed in (42, 43)]
print('queued:', [j['name'] for j in jobs])


In [ ]:
codes = run_jobs(jobs)
print('exit codes:', codes)
print(f'elapsed {(time.time() - T_START) / 3600:.1f} h')


In [ ]:
def load_probs(name):
    run = RUNS / name
    p = run / f'{TEST_CSV.stem}_probs.npy'
    ids = run / f'{TEST_CSV.stem}_ids.csv'
    if not p.exists() or not ids.exists():
        raise RuntimeError(f'missing output for {name}: {p}')
    got_ids = pd.read_csv(ids)['id'].tolist()
    if got_ids != test['id'].tolist():
        raise ValueError(f'{name} predictions are not in test-file order')
    probs = np.load(p)
    assert probs.shape == (len(test), len(LABELS)), (name, probs.shape)
    return probs

models = {}
for prompt in ('short', 'defs'):
    for seed in (42, 43):
        name = f'{prompt}_full_s{seed}'
        models[name] = load_probs(name)
        print(name, models[name].shape)

short_mean = np.mean([models['short_full_s42'], models['short_full_s43']], axis=0)
defs_mean = np.mean([models['defs_full_s42'], models['defs_full_s43']], axis=0)
ensemble = np.mean([short_mean, defs_mean], axis=0)

def write_submission(name, probs, zip_name=None):
    pred = pd.DataFrame({
        'id': test['id'],
        'label': [LABELS[i] for i in probs.argmax(1)],
    })
    assert pred['id'].is_unique
    assert set(pred['label']) <= set(LABELS)
    csv_path = SUB / f'predictions_{name}.csv'
    pred.to_csv(csv_path, index=False)
    if zip_name:
        import zipfile
        zip_path = SUB / zip_name
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
            z.write(csv_path, 'predictions.csv')
        print('wrote', zip_path)
    counts = pred['label'].value_counts(normalize=True)
    prior = train[LABEL_COL].value_counts(normalize=True)
    print(name + ' predicted/prior shares: ' +
          '  '.join(f'{c[:5]} {counts.get(c, 0):.1%}/{prior[c]:.1%}' for c in LABELS))
    return pred

for name, probs in [
    ('short_mean', short_mean),
    ('defs_mean', defs_mean),
]:
    np.save(SUB / f'{name}_test_probs.npy', probs)
    write_submission(name, probs)

np.save(SUB / 'prompt_ensemble_test_probs.npy', ensemble)
final_pred = write_submission(
    'prompt_ensemble',
    ensemble,
    'RECOMMENDED_b32_gemma_prompt_ensemble.zip',
)

# Verify that the archive contains exactly one bare predictions.csv.
import zipfile
with zipfile.ZipFile(SUB / 'RECOMMENDED_b32_gemma_prompt_ensemble.zip') as z:
    assert z.namelist() == ['predictions.csv'], z.namelist()

result = {
    'experiment': 'task_b_run32_gemma_prompt_ensemble_full',
    'based_on': 'task_b_run30_gemma_prompt_ensemble_holdout',
    'commit': head,
    'train_rows': len(train),
    'test_rows': len(test),
    'models': list(models),
    'epochs': 3,
    'class_weight': 'balanced',
    'zip': 'RECOMMENDED_b32_gemma_prompt_ensemble.zip',
    'predicted_class_shares': final_pred['label'].value_counts(normalize=True).to_dict(),
    'elapsed_hours': (time.time() - T_START) / 3600,
}
json.dump(result, open(OUT / 'result.json', 'w'), indent=2)
print('Final ZIP:', SUB / 'RECOMMENDED_b32_gemma_prompt_ensemble.zip')
print('No local score is possible: test labels are not released.')
